In [2]:
# ============================================================
# 서울시 역사마스터 ↔ CELL 250m 매핑 FINAL
# - 5km cutoff 적용
# - valid subway area 분리
# - outlier 분리 저장
# ============================================================

import os
import pandas as pd
import numpy as np
from pyproj import Transformer
from scipy.spatial import cKDTree

# =========================
# 0. 경로
# =========================

cell_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"
station_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울시 역사마스터 정보.csv"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final"
os.makedirs(output_dir, exist_ok=True)

all_output_path = os.path.join(
    output_dir,
    "cell_to_nearest_station_ALL.csv"
)

valid_output_path = os.path.join(
    output_dir,
    "cell_to_nearest_station_VALID_5km.csv"
)

outlier_output_path = os.path.join(
    output_dir,
    "cell_to_nearest_station_OUTLIER_5km.csv"
)

# =========================
# 1. CSV 안전 읽기
# =========================

def read_csv_safe(path):

    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:

        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df

        except UnicodeDecodeError:
            continue

    raise ValueError(f"CSV 읽기 실패: {path}")

station_df = read_csv_safe(station_csv_path)
cell_df = read_csv_safe(cell_csv_path)

# =========================
# 2. 컬럼 정리
# =========================

station_df = station_df.rename(columns={
    "역사_ID": "STATION_ID",
    "역사명": "station_name",
    "호선": "line",
    "위도": "lat",
    "경도": "lon"
})

cell_df = cell_df.rename(columns={
    "cell_id": "CELL_ID",
    "Cell_ID": "CELL_ID",
    "x": "CELL_X",
    "y": "CELL_Y",
    "X": "CELL_X",
    "Y": "CELL_Y"
})

station_df = station_df[[
    "STATION_ID",
    "station_name",
    "line",
    "lat",
    "lon"
]].copy()

cell_df = cell_df[[
    "CELL_ID",
    "CELL_X",
    "CELL_Y"
]].copy()

# 숫자형 변환
station_df["lat"] = pd.to_numeric(station_df["lat"], errors="coerce")
station_df["lon"] = pd.to_numeric(station_df["lon"], errors="coerce")

cell_df["CELL_X"] = pd.to_numeric(cell_df["CELL_X"], errors="coerce")
cell_df["CELL_Y"] = pd.to_numeric(cell_df["CELL_Y"], errors="coerce")

# 결측 제거
station_df = station_df.dropna(subset=["lat", "lon"])
cell_df = cell_df.dropna(subset=["CELL_X", "CELL_Y"])

# 중복 제거
station_df = station_df.drop_duplicates(
    subset=["STATION_ID", "station_name", "line", "lat", "lon"]
)

cell_df = cell_df.drop_duplicates(subset=["CELL_ID"])

print("\n[STATION]")
print(station_df.shape)

print("\n[CELL]")
print(cell_df.shape)

# =========================
# 3. WGS84 → EPSG:5179
# =========================

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:5179",
    always_xy=True
)

station_x, station_y = transformer.transform(
    station_df["lon"].values,
    station_df["lat"].values
)

station_df["station_x_5179"] = station_x
station_df["station_y_5179"] = station_y

# =========================
# 4. KDTree nearest station
# =========================

station_points = station_df[[
    "station_x_5179",
    "station_y_5179"
]].to_numpy()

cell_points = cell_df[[
    "CELL_X",
    "CELL_Y"
]].to_numpy()

tree = cKDTree(station_points)

distances, indices = tree.query(cell_points, k=1)

nearest_station = (
    station_df
    .iloc[indices]
    .reset_index(drop=True)
)

# =========================
# 5. 최종 매핑
# =========================

mapped = cell_df.reset_index(drop=True).copy()

mapped["STATION_ID"] = nearest_station["STATION_ID"].values
mapped["nearest_station"] = nearest_station["station_name"].values
mapped["nearest_line"] = nearest_station["line"].values

mapped["station_lat"] = nearest_station["lat"].values
mapped["station_lon"] = nearest_station["lon"].values

mapped["station_x_5179"] = nearest_station["station_x_5179"].values
mapped["station_y_5179"] = nearest_station["station_y_5179"].values

mapped["nearest_station_distance_m"] = distances

# =========================
# 6. 접근성 feature
# =========================

mapped["station_access_250m"] = (
    mapped["nearest_station_distance_m"] <= 250
)

mapped["station_access_500m"] = (
    mapped["nearest_station_distance_m"] <= 500
)

mapped["station_access_1km"] = (
    mapped["nearest_station_distance_m"] <= 1000
)

mapped["station_access_2km"] = (
    mapped["nearest_station_distance_m"] <= 2000
)

mapped["station_access_5km"] = (
    mapped["nearest_station_distance_m"] <= 5000
)

# 핵심 cutoff
mapped["valid_station_mapping"] = (
    mapped["nearest_station_distance_m"] <= 5000
)

# accessibility score
mapped["station_accessibility_score"] = (
    1 / (mapped["nearest_station_distance_m"] + 1)
)

# =========================
# 7. distance group
# =========================

mapped["distance_group"] = pd.cut(
    mapped["nearest_station_distance_m"],
    bins=[0, 250, 500, 1000, 2000, 5000, np.inf],
    labels=[
        "0-250m",
        "250-500m",
        "500m-1km",
        "1-2km",
        "2-5km",
        "5km+"
    ],
    include_lowest=True
)

# =========================
# 8. valid / outlier 분리
# =========================

mapped_valid = mapped[
    mapped["valid_station_mapping"]
].copy()

mapped_outlier = mapped[
    ~mapped["valid_station_mapping"]
].copy()

# outlier는 역명 제거
mapped_outlier["nearest_station"] = np.nan
mapped_outlier["nearest_line"] = np.nan
mapped_outlier["STATION_ID"] = np.nan

# =========================
# 9. 저장
# =========================

mapped.to_csv(
    all_output_path,
    index=False,
    encoding="utf-8-sig"
)

mapped_valid.to_csv(
    valid_output_path,
    index=False,
    encoding="utf-8-sig"
)

mapped_outlier.to_csv(
    outlier_output_path,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 10. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")

print("\nALL:")
print(all_output_path)

print("\nVALID <= 5km:")
print(valid_output_path)

print("\nOUTLIER > 5km:")
print(outlier_output_path)

print("\n[DISTANCE SUMMARY]")
print(
    mapped["nearest_station_distance_m"].describe()
)

print("\n[VALID ONLY DISTANCE SUMMARY]")
print(
    mapped_valid["nearest_station_distance_m"].describe()
)

print("\n[ACCESS RATIO]")
print(
    mapped[[
        "station_access_250m",
        "station_access_500m",
        "station_access_1km",
        "station_access_2km",
        "station_access_5km"
    ]].mean()
)

print("\n[DISTANCE GROUP]")
print(
    mapped["distance_group"].value_counts()
)

print("\n[VALID RATIO]")
print(
    mapped["valid_station_mapping"].mean()
)

print("\n[TOP 20 CLOSE CELLS]")
print(
    mapped_valid
    .sort_values(
        "nearest_station_distance_m",
        ascending=True
    )
    .head(20)[[
        "CELL_ID",
        "nearest_station",
        "nearest_line",
        "nearest_station_distance_m"
    ]]
)

print("\n[TOP 20 FAR OUTLIERS]")
print(
    mapped_outlier
    .sort_values(
        "nearest_station_distance_m",
        ascending=False
    )
    .head(20)[[
        "CELL_ID",
        "nearest_station_distance_m"
    ]]
)

print("\n[UNIQUE VALID STATIONS]")
print(
    mapped_valid["STATION_ID"].nunique()
)

read success: 서울시 역사마스터 정보.csv / cp949
read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig

[STATION]
(783, 5)

[CELL]
(207205, 3)

DONE

ALL:
C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_ALL.csv

VALID <= 5km:
C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv

OUTLIER > 5km:
C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_OUTLIER_5km.csv

[DISTANCE SUMMARY]
count    207205.000000
mean      10199.882088
std       16214.805120
min           7.297703
25%        2540.151201
50%        6717.861391
75%       13220.086260
max      171420.444168
Name: nearest_station_distance_m, dtype: float64

[VALID ONLY DISTANCE SUMMARY]
count    85543.000000
mean      2235.257249
std       1358.430363
min          7.297703
25%       1057.838784
50%       2052.468900
75%       3326.828707
max       4999.976183
Name: nearest_station_distance_m, dtype: float64

[ACCESS RATIO]
station_access_250m   

In [3]:
# ============================================================
# TEST PIPELINE
# 최종 역-cell 매핑 VALID_5km 사용
# mobility 1개 + 30min station 1개 + station daily 1개 테스트
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로 설정
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\research_test_valid_mapping"
os.makedirs(output_dir, exist_ok=True)

# 최종 매핑 파일
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

# mobility cell 데이터 1개
mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

# 30분 station stats 1개
station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

# station daily aggregate 1개
station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

final_output_path = os.path.join(output_dir, "TEST_final_valid_mapping_dataset.csv")
top_hidden_output_path = os.path.join(output_dir, "TEST_top_hidden_demand.csv")
station_summary_output_path = os.path.join(output_dir, "TEST_station_summary.csv")


# =========================
# 1. 유틸 함수
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)

    if s.max() == s.min():
        return pd.Series(0, index=s.index)

    return (s - s.min()) / (s.max() - s.min())


# =========================
# 2. 최종 mapping 로드
# =========================

mapping = read_csv_safe(mapping_path)

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

mapping_small = mapping[[
    "CELL_ID",
    "STATION_ID",
    "nearest_station",
    "nearest_line",
    "nearest_station_distance_m",
    "station_access_250m",
    "station_access_500m",
    "station_access_1km",
    "station_access_2km",
    "station_access_5km",
    "station_accessibility_score",
    "distance_group",
    "valid_station_mapping"
]].copy()

mapping_small = mapping_small.drop_duplicates(subset=["CELL_ID"])

print("\n[MAPPING SMALL]")
print(mapping_small.shape)
print(mapping_small.head())


# =========================
# 3. mobility cell 1개 로드 + cell 단위 집계
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.columns.tolist())
print(mobility.head())

needed_mobility_cols = [
    "datetime",
    "CELL_ID_BASE",
    "out_cnt",
    "in_cnt",
    "net_cnt"
]

missing = [c for c in needed_mobility_cols if c not in mobility.columns]

if missing:
    raise ValueError(f"mobility 컬럼 없음: {missing}")

mobility = mobility[needed_mobility_cols].copy()

mobility = mobility.rename(columns={
    "CELL_ID_BASE": "CELL_ID",
    "out_cnt": "mobility_out_cnt",
    "in_cnt": "mobility_in_cnt",
    "net_cnt": "mobility_net_cnt"
})

mobility["mobility_out_cnt"] = pd.to_numeric(mobility["mobility_out_cnt"], errors="coerce").fillna(0)
mobility["mobility_in_cnt"] = pd.to_numeric(mobility["mobility_in_cnt"], errors="coerce").fillna(0)
mobility["mobility_net_cnt"] = pd.to_numeric(mobility["mobility_net_cnt"], errors="coerce").fillna(0)

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("mobility_out_cnt", "sum"),
        mobility_in_sum=("mobility_in_cnt", "sum"),
        mobility_net_sum=("mobility_net_cnt", "sum"),
        mobility_out_mean=("mobility_out_cnt", "mean"),
        mobility_in_mean=("mobility_in_cnt", "mean"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = (
    cell_mobility["mobility_net_sum"].abs()
)

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# =========================
# 4. 30분 station stats 로드 + station 단위 집계
# =========================

station_30 = pd.read_parquet(station_30min_path)

print("\n[STATION 30MIN RAW]")
print(station_30.shape)
print(station_30.columns.tolist())
print(station_30.head())

for col in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_30[col] = pd.to_numeric(station_30[col], errors="coerce").fillna(0)

station_30_agg = (
    station_30
    .groupby("STATION_ID", as_index=False)
    .agg(
        station_30_out_sum=("od_out_cnt", "sum"),
        station_30_in_sum=("od_in_cnt", "sum"),
        station_30_net_sum=("od_net_cnt", "sum"),
        station_30_out_mean=("od_out_cnt", "mean"),
        station_30_in_mean=("od_in_cnt", "mean"),
        station_30_record_count=("datetime", "count")
    )
)

station_30_agg["station_30_total_activity"] = (
    station_30_agg["station_30_out_sum"] +
    station_30_agg["station_30_in_sum"]
)

print("\n[STATION 30MIN AGG]")
print(station_30_agg.shape)
print(station_30_agg.head())


# =========================
# 5. station daily aggregate 로드 + station 단위 집계
# =========================

station_daily = pd.read_parquet(station_daily_path)

print("\n[STATION DAILY RAW]")
print(station_daily.shape)
print(station_daily.columns.tolist())
print(station_daily.head())

for col in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    station_daily[col] = pd.to_numeric(station_daily[col], errors="coerce").fillna(0)

station_daily_agg = (
    station_daily
    .groupby("STATION_ID", as_index=False)
    .agg(
        station_daily_out_sum=("od_out_cnt", "sum"),
        station_daily_in_sum=("od_in_cnt", "sum"),
        station_daily_net_sum=("od_net_cnt", "sum"),
        station_daily_record_count=("datetime", "count")
    )
)

station_daily_agg["station_daily_total_activity"] = (
    station_daily_agg["station_daily_out_sum"] +
    station_daily_agg["station_daily_in_sum"]
)

print("\n[STATION DAILY AGG]")
print(station_daily_agg.shape)
print(station_daily_agg.head())


# =========================
# 6. station activity 통합
# =========================

station_activity = station_30_agg.merge(
    station_daily_agg,
    on="STATION_ID",
    how="outer"
).fillna(0)

station_activity["station_total_activity"] = (
    station_activity["station_30_total_activity"] +
    station_activity["station_daily_total_activity"]
)

station_activity["station_total_in"] = (
    station_activity["station_30_in_sum"] +
    station_activity["station_daily_in_sum"]
)

station_activity["station_total_out"] = (
    station_activity["station_30_out_sum"] +
    station_activity["station_daily_out_sum"]
)

station_activity["station_total_net"] = (
    station_activity["station_total_in"] -
    station_activity["station_total_out"]
)

print("\n[STATION ACTIVITY]")
print(station_activity.shape)
print(station_activity.head())


# =========================
# 7. cell mobility + valid mapping merge
# =========================

final = cell_mobility.merge(
    mapping_small,
    on="CELL_ID",
    how="inner"
)

print("\n[MERGE 1: MOBILITY + MAPPING]")
print(final.shape)
print(final.head())

print("\n[MAPPING MATCH CHECK]")
print("mobility cell 수:", cell_mobility["CELL_ID"].nunique())
print("valid mapping cell 수:", mapping_small["CELL_ID"].nunique())
print("merge 후 cell 수:", final["CELL_ID"].nunique())


# =========================
# 8. station activity merge
# =========================

final = final.merge(
    station_activity,
    on="STATION_ID",
    how="left"
)

station_cols = [c for c in final.columns if c.startswith("station_")]

for c in station_cols:
    if final[c].dtype != "object":
        final[c] = final[c].fillna(0)

print("\n[FINAL MERGED]")
print(final.shape)
print(final.head())


# =========================
# 9. 연구용 feature 생성
# =========================

final["actual_total_flow_norm"] = normalize_minmax(final["actual_total_flow"])
final["mobility_out_norm"] = normalize_minmax(final["mobility_out_sum"])
final["mobility_in_norm"] = normalize_minmax(final["mobility_in_sum"])

final["nearest_station_distance_m"] = pd.to_numeric(
    final["nearest_station_distance_m"],
    errors="coerce"
)

final["station_accessibility_score"] = pd.to_numeric(
    final["station_accessibility_score"],
    errors="coerce"
).fillna(0)

final["accessibility_norm"] = normalize_minmax(
    final["station_accessibility_score"]
)

final["distance_norm"] = normalize_minmax(
    final["nearest_station_distance_m"]
)

final["station_activity_norm"] = normalize_minmax(
    final["station_total_activity"]
)

# 핵심 1:
# 실제 이동량은 높은데, 역 접근성이 낮은 cell
final["hidden_demand_score"] = (
    final["actual_total_flow_norm"] *
    (1 - final["accessibility_norm"])
)

# 핵심 2:
# 실제 이동량은 높은데, 역까지 거리가 먼 cell
final["distance_weighted_hidden_demand"] = (
    final["actual_total_flow_norm"] *
    final["distance_norm"]
)

# 핵심 3:
# 실제 이동량은 높은데, station activity representation은 낮은 cell
final["subway_representation_gap"] = (
    final["actual_total_flow_norm"] -
    final["station_activity_norm"]
)

# 핵심 4:
# 역 활동량은 높은데 실제 mobility는 낮은 경우
final["subway_overrepresentation_score"] = (
    final["station_activity_norm"] *
    (1 - final["actual_total_flow_norm"])
)

# 역세권 + 실제 이동량 높은 지역
final["station_area_mobility_score"] = (
    final["actual_total_flow_norm"] *
    final["accessibility_norm"]
)


# =========================
# 10. area type 분류
# =========================

q_hidden = final["hidden_demand_score"].quantile(0.90)
q_gap = final["subway_representation_gap"].quantile(0.90)
q_station_area = final["station_area_mobility_score"].quantile(0.90)

conditions = [
    final["hidden_demand_score"] >= q_hidden,
    final["subway_representation_gap"] >= q_gap,
    final["station_area_mobility_score"] >= q_station_area
]

choices = [
    "hidden_demand_area",
    "subway_under_represented",
    "station_area_high_mobility"
]

final["area_type"] = np.select(
    conditions,
    choices,
    default="normal"
)


# =========================
# 11. station summary
# =========================

station_summary = (
    final
    .groupby(["STATION_ID", "nearest_station", "nearest_line"], as_index=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        total_actual_flow=("actual_total_flow", "sum"),
        mean_actual_flow=("actual_total_flow", "mean"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        mean_hidden_demand=("hidden_demand_score", "mean"),
        max_hidden_demand=("hidden_demand_score", "max"),
        mean_representation_gap=("subway_representation_gap", "mean"),
        hidden_cell_count=("area_type", lambda x: (x == "hidden_demand_area").sum())
    )
)

station_summary["hidden_cell_ratio"] = (
    station_summary["hidden_cell_count"] /
    station_summary["cell_count"]
)

station_summary = station_summary.sort_values(
    ["hidden_cell_count", "total_actual_flow"],
    ascending=False
)


# =========================
# 12. 저장
# =========================

final.to_csv(
    final_output_path,
    index=False,
    encoding="utf-8-sig"
)

top_hidden = (
    final
    .sort_values("hidden_demand_score", ascending=False)
    .head(200)
)

top_hidden.to_csv(
    top_hidden_output_path,
    index=False,
    encoding="utf-8-sig"
)

station_summary.to_csv(
    station_summary_output_path,
    index=False,
    encoding="utf-8-sig"
)


# =========================
# 13. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")

print("\n저장 파일")
print("final:", final_output_path)
print("top hidden:", top_hidden_output_path)
print("station summary:", station_summary_output_path)

print("\n[FINAL SHAPE]")
print(final.shape)

print("\n[KEY METRICS]")
print(final[[
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "actual_total_flow_norm",
    "accessibility_norm",
    "station_activity_norm",
    "hidden_demand_score",
    "distance_weighted_hidden_demand",
    "subway_representation_gap",
    "subway_overrepresentation_score",
    "station_area_mobility_score"
]].describe())

print("\n[AREA TYPE COUNTS]")
print(final["area_type"].value_counts())

print("\n[TOP HIDDEN DEMAND]")
print(top_hidden[[
    "CELL_ID",
    "STATION_ID",
    "nearest_station",
    "nearest_line",
    "actual_total_flow",
    "nearest_station_distance_m",
    "station_total_activity",
    "hidden_demand_score",
    "subway_representation_gap",
    "area_type"
]].head(30))

print("\n[STATION SUMMARY TOP]")
print(station_summary.head(30))

print("\n[DISTANCE GROUP SUMMARY]")
distance_group_summary = (
    final
    .groupby("distance_group", observed=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        total_actual_flow=("actual_total_flow", "sum"),
        mean_actual_flow=("actual_total_flow", "mean"),
        mean_hidden_demand=("hidden_demand_score", "mean"),
        mean_station_activity=("station_total_activity", "mean")
    )
    .reset_index()
)

print(distance_group_summary)


[MAPPING]
(85543, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group']
      CELL_ID  CELL_X   CELL_Y  STATION_ID nearest_station nearest_line  \
0  다사31254225  931375  1942375        3121              동수        인천1호선   
1  다사33752875  933875  1928875        1761              정왕          안산선   
2  다사53005725  953125  1957375        2614             독바위          6호선   
3  다사53255000  953375  1950125         426             서울역          4호선   
4  다아58500350  958625  2003625        1918              전곡          경원선   

   station_lat  station_lon  station_x_5179  station_y_5179  \
0    37.485312   126.718247   930886.700124    1.943184e+06   
1    37.351735   126.742989   932955.

In [ ]:
import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================
bridge_csv_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울교통공사_역명 지하철역 검색.csv"

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

station_30min_path = r"C:\Users\82108\Desktop\새 폴더 (8)\30_od_output (1)\30_od_output\TBDM_TRANSIT_STAT_SUBWAY_20170102_station_stats.parquet"

station_daily_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191001_station.parquet"

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\station_id_bridge_output"
os.makedirs(output_dir, exist_ok=True)

bridge_output_path = os.path.join(output_dir, "station_id_bridge_checked.csv")
mapping_fixed_output_path = os.path.join(output_dir, "cell_to_nearest_station_VALID_5km_ID_FIXED.csv")

# =========================
# 1. 안전 읽기
# =========================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

bridge = read_csv_safe(bridge_csv_path)
mapping = pd.read_csv(mapping_path, encoding="utf-8-sig")

station_30 = pd.read_parquet(station_30min_path)
station_daily = pd.read_parquet(station_daily_path)

print("\n[BRIDGE RAW]")
print(bridge.shape)
print(bridge.columns.tolist())
print(bridge.head())

print("\n[MAPPING RAW]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

print("\n[STATION 30]")
print(station_30.shape)
print(station_30.columns.tolist())

print("\n[STATION DAILY]")
print(station_daily.shape)
print(station_daily.columns.tolist())

# =========================
# 2. bridge 정리
# =========================
bridge = bridge.rename(columns={
    "전철역코드": "subway_station_code",
    "전철역명": "station_name",
    "호선": "line",
    "외부코드": "external_code"
})

bridge["station_name"] = bridge["station_name"].astype(str).str.strip()
bridge["line"] = bridge["line"].astype(str).str.strip()
bridge["external_code"] = bridge["external_code"].astype(str).str.strip()

# 외부코드가 숫자인 경우만 station_stats STATION_ID 후보로 사용
bridge["STATION_ID_CANDIDATE"] = pd.to_numeric(
    bridge["external_code"],
    errors="coerce"
)

bridge_numeric = bridge.dropna(subset=["STATION_ID_CANDIDATE"]).copy()
bridge_numeric["STATION_ID_CANDIDATE"] = bridge_numeric["STATION_ID_CANDIDATE"].astype(int)

print("\n[BRIDGE NUMERIC]")
print(bridge_numeric.shape)
print(bridge_numeric.head())

# =========================
# 3. station stats ID와 실제 겹침 검증
# =========================
station_30_ids = set(pd.to_numeric(station_30["STATION_ID"], errors="coerce").dropna().astype(int))
station_daily_ids = set(pd.to_numeric(station_daily["STATION_ID"], errors="coerce").dropna().astype(int))
bridge_ids = set(bridge_numeric["STATION_ID_CANDIDATE"].unique())

print("\n[ID OVERLAP CHECK]")
print("bridge numeric ids:", len(bridge_ids))
print("30min ids:", len(station_30_ids))
print("daily ids:", len(station_daily_ids))
print("bridge ∩ 30min:", len(bridge_ids & station_30_ids))
print("bridge ∩ daily:", len(bridge_ids & station_daily_ids))
print("30min ∩ daily:", len(station_30_ids & station_daily_ids))

print("\n겹치는 ID 예시:")
print(sorted(list(bridge_ids & station_30_ids))[:30])

# =========================
# 4. bridge table 확정
# =========================
valid_stats_ids = station_30_ids | station_daily_ids

bridge_checked = bridge_numeric[
    bridge_numeric["STATION_ID_CANDIDATE"].isin(valid_stats_ids)
].copy()

bridge_checked = bridge_checked.rename(columns={
    "STATION_ID_CANDIDATE": "STATS_STATION_ID"
})

bridge_checked = bridge_checked[[
    "STATS_STATION_ID",
    "station_name",
    "line",
    "subway_station_code",
    "external_code"
]].drop_duplicates()

bridge_checked.to_csv(
    bridge_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n[BRIDGE CHECKED]")
print(bridge_checked.shape)
print(bridge_checked.head())
print("saved:", bridge_output_path)

# =========================
# 5. 기존 mapping의 역사마스터 STATION_ID는 보존
#    대신 station_stats에 맞는 STATS_STATION_ID를 새로 붙임
# =========================

mapping = mapping.rename(columns={
    "STATION_ID": "MASTER_STATION_ID"
})

# 이름 + 호선 기준으로 우선 매칭
# 호선 표기 차이가 있을 수 있으므로 먼저 역명만으로도 후보 생성
mapping["nearest_station_clean"] = mapping["nearest_station"].astype(str).str.strip()
mapping["nearest_line_clean"] = mapping["nearest_line"].astype(str).str.strip()

bridge_checked["station_name_clean"] = bridge_checked["station_name"].astype(str).str.strip()
bridge_checked["line_clean"] = bridge_checked["line"].astype(str).str.strip()

# 5-1. 역명 + 호선 exact
fixed = mapping.merge(
    bridge_checked,
    left_on=["nearest_station_clean", "nearest_line_clean"],
    right_on=["station_name_clean", "line_clean"],
    how="left"
)

# 5-2. exact 실패분은 역명만으로 재시도
unmatched_mask = fixed["STATS_STATION_ID"].isna()

name_only_bridge = (
    bridge_checked
    .sort_values("STATS_STATION_ID")
    .drop_duplicates(subset=["station_name_clean"])
)

retry = mapping.loc[unmatched_mask].merge(
    name_only_bridge,
    left_on="nearest_station_clean",
    right_on="station_name_clean",
    how="left"
)

# retry 결과를 fixed에 반영
cols_to_fill = [
    "STATS_STATION_ID",
    "station_name",
    "line",
    "subway_station_code",
    "external_code",
    "station_name_clean",
    "line_clean"
]

for col in cols_to_fill:
    if col in fixed.columns and col in retry.columns:
        fixed.loc[unmatched_mask, col] = retry[col].values

# =========================
# 6. 매칭 품질 확인
# =========================
print("\n[FIXED MAPPING CHECK]")
print("전체 mapping rows:", len(fixed))
print("STATS_STATION_ID matched:", fixed["STATS_STATION_ID"].notna().sum())
print("unmatched:", fixed["STATS_STATION_ID"].isna().sum())
print("match ratio:", fixed["STATS_STATION_ID"].notna().mean())

print("\n[UNMATCHED SAMPLE]")
print(
    fixed[fixed["STATS_STATION_ID"].isna()]
    [["MASTER_STATION_ID", "nearest_station", "nearest_line"]]
    .drop_duplicates()
    .head(50)
)

print("\n[MATCHED SAMPLE]")
print(
    fixed[fixed["STATS_STATION_ID"].notna()]
    [["MASTER_STATION_ID", "nearest_station", "nearest_line", "STATS_STATION_ID", "station_name", "line", "external_code"]]
    .drop_duplicates()
    .head(50)
)

# =========================
# 7. 저장
# =========================
fixed.to_csv(
    mapping_fixed_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")
print("bridge saved:", bridge_output_path)
print("fixed mapping saved:", mapping_fixed_output_path)

read success: 서울교통공사_역명 지하철역 검색.csv / cp949

[BRIDGE RAW]
(799, 4)
['전철역코드', '전철역명', '호선', '외부코드']
  전철역코드       전철역명    호선  외부코드
0  101C         회기   경의선  K118
1  0205  동대문역사문화공원  02호선   205
2  0206         신당  02호선   206
3  0207       상왕십리  02호선   207
4  0208        왕십리  02호선   208

[MAPPING RAW]
(85543, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group']
      CELL_ID  CELL_X   CELL_Y  STATION_ID nearest_station nearest_line  \
0  다사31254225  931375  1942375        3121              동수        인천1호선   
1  다사33752875  933875  1928875        1761              정왕          안산선   
2  다사53005725  953125  1957375        2614             독바위          6호선   
3  다사53255000  953375  19

: 

In [1]:
import pandas as pd
import os

# =========================================================
# 경로
# =========================================================
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울시 역사마스터 정보.csv"

output_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

# =========================================================
# CSV 안전 로드
# =========================================================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except:
            continue
    raise ValueError(f"cannot read: {path}")

# =========================================================
# 로드
# =========================================================
mapping = read_csv_safe(mapping_path)
master = read_csv_safe(master_path)

# =========================================================
# 컬럼 확인
# =========================================================
print("\n[MAPPING]")
print(mapping.shape)
print(mapping.columns.tolist())

print("\n[MASTER]")
print(master.shape)
print(master.columns.tolist())

# =========================================================
# master 정리
# =========================================================
master_clean = master[[
    "역사_ID",
    "역사명",
    "호선"
]].copy()

master_clean.columns = [
    "MASTER_STATION_ID",
    "master_station_name",
    "master_line"
]

# 문자열 정리
master_clean["master_station_name"] = (
    master_clean["master_station_name"]
    .astype(str)
    .str.strip()
)

master_clean["master_line"] = (
    master_clean["master_line"]
    .astype(str)
    .str.strip()
)

# =========================================================
# mapping 정리
# =========================================================
mapping["nearest_station"] = (
    mapping["nearest_station"]
    .astype(str)
    .str.strip()
)

mapping["nearest_line"] = (
    mapping["nearest_line"]
    .astype(str)
    .str.strip()
)

# =========================================================
# 이름 + 호선 기준 merge
# =========================================================
fixed = mapping.merge(
    master_clean,
    left_on=["nearest_station", "nearest_line"],
    right_on=["master_station_name", "master_line"],
    how="left"
)

# =========================================================
# 기존 STATION_ID 제거
# =========================================================
if "STATION_ID" in fixed.columns:
    fixed = fixed.drop(columns=["STATION_ID"])

# =========================================================
# MASTER_STATION_ID를 앞으로 이동
# =========================================================
cols = fixed.columns.tolist()

new_cols = (
    ["MASTER_STATION_ID"]
    + [c for c in cols if c != "MASTER_STATION_ID"]
)

fixed = fixed[new_cols]

# =========================================================
# 결과 확인
# =========================================================
print("\n[RESULT]")
print(fixed.shape)

print("\n[MATCH CHECK]")
print("matched:",
      fixed["MASTER_STATION_ID"].notna().sum())

print("unmatched:",
      fixed["MASTER_STATION_ID"].isna().sum())

print("match ratio:",
      fixed["MASTER_STATION_ID"].notna().mean())

print("\n[HEAD]")
print(
    fixed[[
        "MASTER_STATION_ID",
        "nearest_station",
        "nearest_line"
    ]].head(20)
)

# =========================================================
# 저장
# =========================================================
fixed.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")
print("saved:", output_path)

read success: cell_to_nearest_station_VALID_5km.csv / utf-8-sig
read success: 서울시 역사마스터 정보.csv / cp949

[MAPPING]
(85543, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group']

[MASTER]
(783, 5)
['역사_ID', '역사명', '호선', '위도', '경도']

[RESULT]
(85543, 21)

[MATCH CHECK]
matched: 85543
unmatched: 0
match ratio: 1.0

[HEAD]
    MASTER_STATION_ID nearest_station nearest_line
0                3121              동수        인천1호선
1                1761              정왕          안산선
2                2614             독바위          6호선
3                 426             서울역          4호선
4                1918              전곡          경원선
5                1848          압구정로데오          분당선
6         

In [2]:
import pandas as pd
import os

# =========================================================
# 경로
# =========================================================
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km.csv"

master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\서울시 역사마스터 정보.csv"

output_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

# =========================================================
# CSV 안전 로드
# =========================================================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except:
            continue
    raise ValueError(f"cannot read: {path}")

# =========================================================
# 로드
# =========================================================
mapping = read_csv_safe(mapping_path)
master = read_csv_safe(master_path)

# =========================================================
# 컬럼 확인
# =========================================================
print("\n[MAPPING]")
print(mapping.shape)
print(mapping.columns.tolist())

print("\n[MASTER]")
print(master.shape)
print(master.columns.tolist())

# =========================================================
# master 정리
# =========================================================
master_clean = master[[
    "역사_ID",
    "역사명",
    "호선"
]].copy()

master_clean.columns = [
    "MASTER_STATION_ID",
    "master_station_name",
    "master_line"
]

# 문자열 정리
master_clean["master_station_name"] = (
    master_clean["master_station_name"]
    .astype(str)
    .str.strip()
)

master_clean["master_line"] = (
    master_clean["master_line"]
    .astype(str)
    .str.strip()
)

# =========================================================
# mapping 정리
# =========================================================
mapping["nearest_station"] = (
    mapping["nearest_station"]
    .astype(str)
    .str.strip()
)

mapping["nearest_line"] = (
    mapping["nearest_line"]
    .astype(str)
    .str.strip()
)

# =========================================================
# 이름 + 호선 기준 merge
# =========================================================
fixed = mapping.merge(
    master_clean,
    left_on=["nearest_station", "nearest_line"],
    right_on=["master_station_name", "master_line"],
    how="left"
)

# =========================================================
# 기존 STATION_ID 제거
# =========================================================
if "STATION_ID" in fixed.columns:
    fixed = fixed.drop(columns=["STATION_ID"])

# =========================================================
# MASTER_STATION_ID를 앞으로 이동
# =========================================================
cols = fixed.columns.tolist()

new_cols = (
    ["MASTER_STATION_ID"]
    + [c for c in cols if c != "MASTER_STATION_ID"]
)

fixed = fixed[new_cols]

# =========================================================
# 결과 확인
# =========================================================
print("\n[RESULT]")
print(fixed.shape)

print("\n[MATCH CHECK]")
print("matched:",
      fixed["MASTER_STATION_ID"].notna().sum())

print("unmatched:",
      fixed["MASTER_STATION_ID"].isna().sum())

print("match ratio:",
      fixed["MASTER_STATION_ID"].notna().mean())

print("\n[HEAD]")
print(
    fixed[[
        "MASTER_STATION_ID",
        "nearest_station",
        "nearest_line"
    ]].head(20)
)

# =========================================================
# 저장
# =========================================================
fixed.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")
print("saved:", output_path)

read success: cell_to_nearest_station_VALID_5km.csv / utf-8-sig
read success: 서울시 역사마스터 정보.csv / cp949

[MAPPING]
(85543, 19)
['CELL_ID', 'CELL_X', 'CELL_Y', 'STATION_ID', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group']

[MASTER]
(783, 5)
['역사_ID', '역사명', '호선', '위도', '경도']

[RESULT]
(85543, 21)

[MATCH CHECK]
matched: 85543
unmatched: 0
match ratio: 1.0

[HEAD]
    MASTER_STATION_ID nearest_station nearest_line
0                3121              동수        인천1호선
1                1761              정왕          안산선
2                2614             독바위          6호선
3                 426             서울역          4호선
4                1918              전곡          경원선
5                1848          압구정로데오          분당선
6         

In [3]:
# ============================================================
# CELL-level analysis
# 사용:
# 1) mobility parquet
# 2) cell_to_nearest_station_VALID_5km_MASTER_ID.csv
#
# 주의:
# CELL_ID_BASE가 실제 CELL_ID와 같은 경우만 우선 merge
# 숫자 index 복원은 여기서 하지 않음
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\cell_level_analysis"
os.makedirs(output_dir, exist_ok=True)

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

output_path = os.path.join(output_dir, "CELL_LEVEL_TEST_merged.csv")
top_output_path = os.path.join(output_dir, "CELL_LEVEL_TEST_top_cells.csv")
check_output_path = os.path.join(output_dir, "CELL_LEVEL_TEST_merge_check.csv")

# =========================
# 1. 유틸
# =========================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

# =========================
# 2. mapping 로드
# =========================

mapping = read_csv_safe(mapping_path)

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()

print("\n[MAPPING]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

# =========================
# 3. mobility 로드
# =========================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.columns.tolist())
print(mobility.head())

# =========================
# 4. mobility CELL_ID_BASE를 문자열 CELL_ID 후보로 사용
# =========================

mobility["CELL_ID"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)

# =========================
# 5. CELL 단위 mobility 집계
# =========================

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] +
    cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = (
    cell_mobility["mobility_net_sum"].abs()
)

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())

# =========================
# 6. merge check
# =========================

mobility_ids = set(cell_mobility["CELL_ID"])
mapping_ids = set(mapping["CELL_ID"])

overlap = mobility_ids & mapping_ids

check = pd.DataFrame([{
    "mobility_unique_cell": len(mobility_ids),
    "mapping_unique_cell": len(mapping_ids),
    "overlap_cell": len(overlap),
    "overlap_ratio_vs_mobility": len(overlap) / len(mobility_ids) if len(mobility_ids) else 0,
    "overlap_ratio_vs_mapping": len(overlap) / len(mapping_ids) if len(mapping_ids) else 0,
}])

check.to_csv(check_output_path, index=False, encoding="utf-8-sig")

print("\n[MERGE CHECK]")
print(check)

# =========================
# 7. CELL-level merge
# =========================

cell_level = mapping.merge(
    cell_mobility,
    on="CELL_ID",
    how="left"
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_level[c] = pd.to_numeric(cell_level[c], errors="coerce").fillna(0)

# =========================
# 8. CELL-level 지표
# =========================

cell_level["actual_total_flow_norm"] = normalize_minmax(
    cell_level["actual_total_flow"]
)

cell_level["distance_norm"] = normalize_minmax(
    cell_level["nearest_station_distance_m"]
)

cell_level["accessibility_norm"] = normalize_minmax(
    cell_level["station_accessibility_score"]
)

# 이동량은 높은데 역 접근성은 낮은 cell
cell_level["cell_hidden_demand_score"] = (
    cell_level["actual_total_flow_norm"] *
    (1 - cell_level["accessibility_norm"])
)

# 이동량 높고 역과 거리가 먼 cell
cell_level["distance_weighted_demand"] = (
    cell_level["actual_total_flow_norm"] *
    cell_level["distance_norm"]
)

# =========================
# 9. 저장
# =========================

cell_level.to_csv(output_path, index=False, encoding="utf-8-sig")

top_cells = (
    cell_level
    .sort_values("cell_hidden_demand_score", ascending=False)
    .head(200)
)

top_cells.to_csv(top_output_path, index=False, encoding="utf-8-sig")

# =========================
# 10. 결과 확인
# =========================

print("\n================================================")
print("DONE")
print("================================================")
print("merged:", output_path)
print("top cells:", top_output_path)
print("check:", check_output_path)

print("\n[CELL LEVEL SHAPE]")
print(cell_level.shape)

print("\n[KEY METRICS]")
print(cell_level[[
    "actual_total_flow",
    "nearest_station_distance_m",
    "actual_total_flow_norm",
    "distance_norm",
    "accessibility_norm",
    "cell_hidden_demand_score",
    "distance_weighted_demand"
]].describe())

print("\n[TOP CELLS]")
print(top_cells[[
    "CELL_ID",
    "MASTER_STATION_ID",
    "nearest_station",
    "nearest_line",
    "distance_group",
    "actual_total_flow",
    "nearest_station_distance_m",
    "cell_hidden_demand_score",
    "distance_weighted_demand"
]].head(30))

read success: cell_to_nearest_station_VALID_5km_MASTER_ID.csv / utf-8-sig

[MAPPING]
(85543, 21)
['MASTER_STATION_ID', 'CELL_ID', 'CELL_X', 'CELL_Y', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group', 'master_station_name', 'master_line']
   MASTER_STATION_ID     CELL_ID  CELL_X   CELL_Y nearest_station  \
0               3121  다사31254225  931375  1942375              동수   
1               1761  다사33752875  933875  1928875              정왕   
2               2614  다사53005725  953125  1957375             독바위   
3                426  다사53255000  953375  1950125             서울역   
4               1918  다아58500350  958625  2003625              전곡   

  nearest_line  station_lat  station_lon  station_x_5179  station_y_5179  ...  \

In [4]:
import os
import pandas as pd

# =========================
# 경로
# =========================
output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\missing_cell_check"
os.makedirs(output_dir, exist_ok=True)

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"
mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

# =========================
# 로드
# =========================
mapping = pd.read_csv(mapping_path, encoding="utf-8-sig")
mobility = pd.read_parquet(mobility_path)

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()
mobility["CELL_ID"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

# =========================
# cell 집합 비교
# =========================
mapping_ids = set(mapping["CELL_ID"])
mobility_ids = set(mobility["CELL_ID"])

overlap_ids = mapping_ids & mobility_ids

# mobility에는 있는데 역세권 mapping에는 없는 cell
mobility_not_in_mapping = mobility_ids - mapping_ids

# 역세권 mapping에는 있는데 mobility에는 없는 cell
mapping_not_in_mobility = mapping_ids - mobility_ids

print("[SUMMARY]")
print("mobility unique cells:", len(mobility_ids))
print("mapping unique cells:", len(mapping_ids))
print("overlap:", len(overlap_ids))
print("mobility not in mapping:", len(mobility_not_in_mapping))
print("mapping not in mobility:", len(mapping_not_in_mobility))

# =========================
# 1. mobility에는 있는데 역세권 밖인 cell
# =========================
mobility_cell_flow = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        record_count=("datetime", "count")
    )
)

mobility_cell_flow["actual_total_flow"] = (
    mobility_cell_flow["mobility_out_sum"] +
    mobility_cell_flow["mobility_in_sum"]
)

missing_mobility = mobility_cell_flow[
    mobility_cell_flow["CELL_ID"].isin(mobility_not_in_mapping)
].copy()

missing_mobility = missing_mobility.sort_values(
    "actual_total_flow",
    ascending=False
)

# =========================
# 2. 역세권 mapping에는 있는데 mobility가 없는 cell
# =========================
missing_mapping = mapping[
    mapping["CELL_ID"].isin(mapping_not_in_mobility)
].copy()

missing_mapping = missing_mapping.sort_values(
    ["nearest_station", "nearest_station_distance_m"]
)

# =========================
# 저장
# =========================
missing_mobility_path = os.path.join(output_dir, "mobility_cells_not_in_station_mapping.csv")
missing_mapping_path = os.path.join(output_dir, "station_mapping_cells_without_mobility.csv")

missing_mobility.to_csv(missing_mobility_path, index=False, encoding="utf-8-sig")
missing_mapping.to_csv(missing_mapping_path, index=False, encoding="utf-8-sig")

print("\n[SAVED]")
print("mobility not in mapping:", missing_mobility_path)
print("mapping without mobility:", missing_mapping_path)

print("\n[TOP mobility cells outside station mapping]")
print(missing_mobility.head(30))

print("\n[station mapping cells without mobility sample]")
print(
    missing_mapping[[
        "CELL_ID",
        "MASTER_STATION_ID",
        "nearest_station",
        "nearest_line",
        "nearest_station_distance_m",
        "distance_group"
    ]].head(30)
)

[SUMMARY]
mobility unique cells: 28486
mapping unique cells: 85543
overlap: 22140
mobility not in mapping: 6346
mapping not in mobility: 63403

[SAVED]
mobility not in mapping: C:\Users\82108\Desktop\새 폴더 (8)\missing_cell_check\mobility_cells_not_in_station_mapping.csv
mapping without mobility: C:\Users\82108\Desktop\새 폴더 (8)\missing_cell_check\station_mapping_cells_without_mobility.csv

[TOP mobility cells outside station mapping]
          CELL_ID  mobility_out_sum  mobility_in_sum  mobility_net_sum  \
42          44200          23719.50          24605.0            885.50   
39          44133          22736.00          23159.5            423.50   
56             47          13764.09          12439.0          -1325.09   
3              30          12554.50          12022.5           -532.00   
7           42130          12355.00          11434.5           -920.50   
38          44131          11777.50          11945.5            168.00   
6           42110          12029.50          1

In [5]:
# ============================================================
# MIXED CELL_ID_BASE 처리 버전
# - CELL_ID_BASE가 한글 CELL_ID면 그대로 사용
# - CELL_ID_BASE가 숫자형이면 cell_master index로 복원
# - 복원 후 station mapping과 merge
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================
output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\cell_level_analysis_mixed"
os.makedirs(output_dir, exist_ok=True)

cell_master_path = r"C:\Users\82108\Desktop\새 폴더 (8)\CELL_250x250(UTMK_epsg 5179).csv"

mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

output_path = os.path.join(output_dir, "CELL_LEVEL_MIXED_merged.csv")
top_output_path = os.path.join(output_dir, "CELL_LEVEL_MIXED_top_cells.csv")
missing_output_path = os.path.join(output_dir, "CELL_LEVEL_MIXED_mobility_not_in_mapping.csv")
check_output_path = os.path.join(output_dir, "CELL_LEVEL_MIXED_check.csv")

# =========================
# 1. 유틸
# =========================
def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")

def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

# =========================
# 2. 데이터 로드
# =========================
cell_master = read_csv_safe(cell_master_path)
mapping = read_csv_safe(mapping_path)
mobility = pd.read_parquet(mobility_path)

cell_master = cell_master[["CELL_ID", "CELL_X", "CELL_Y"]].copy()
cell_master = cell_master.reset_index().rename(columns={"index": "CELL_INDEX"})
cell_master["CELL_INDEX"] = cell_master["CELL_INDEX"].astype(int)
cell_master["CELL_ID"] = cell_master["CELL_ID"].astype(str).str.strip()

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()

mobility["CELL_ID_BASE_RAW"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

print("\n[CELL MASTER]", cell_master.shape)
print("[MAPPING]", mapping.shape)
print("[MOBILITY RAW]", mobility.shape)

# =========================
# 3. CELL_ID_BASE 유형 분리
# =========================
mobility["is_korean_cell_id"] = mobility["CELL_ID_BASE_RAW"].str.contains(r"[가-힣]", regex=True)

mobility["CELL_INDEX"] = pd.to_numeric(
    mobility["CELL_ID_BASE_RAW"],
    errors="coerce"
)

print("\n[ID TYPE CHECK]")
print("rows:", len(mobility))
print("korean CELL_ID rows:", mobility["is_korean_cell_id"].sum())
print("numeric-like rows:", (~mobility["is_korean_cell_id"]).sum())
print("CELL_ID_BASE unique:", mobility["CELL_ID_BASE_RAW"].nunique())

# =========================
# 4. 한글 CELL_ID는 그대로 사용
# =========================
mobility_korean = mobility[mobility["is_korean_cell_id"]].copy()
mobility_korean["CELL_ID"] = mobility_korean["CELL_ID_BASE_RAW"]

# 좌표는 cell_master에서 붙임
mobility_korean = mobility_korean.merge(
    cell_master[["CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_ID",
    how="left"
)

# =========================
# 5. 숫자형 CELL_ID_BASE는 cell_master index로 복원
# =========================
mobility_numeric = mobility[~mobility["is_korean_cell_id"]].copy()

valid_idx = (
    mobility_numeric["CELL_INDEX"].notna()
    & (mobility_numeric["CELL_INDEX"] >= 0)
    & (mobility_numeric["CELL_INDEX"] < len(cell_master))
)

mobility_numeric_valid = mobility_numeric[valid_idx].copy()
mobility_numeric_invalid = mobility_numeric[~valid_idx].copy()

mobility_numeric_valid["CELL_INDEX"] = mobility_numeric_valid["CELL_INDEX"].astype(int)

mobility_numeric_valid = mobility_numeric_valid.merge(
    cell_master[["CELL_INDEX", "CELL_ID", "CELL_X", "CELL_Y"]],
    on="CELL_INDEX",
    how="left"
)

print("\n[NUMERIC RESTORE CHECK]")
print("numeric rows:", len(mobility_numeric))
print("numeric valid rows:", len(mobility_numeric_valid))
print("numeric invalid rows:", len(mobility_numeric_invalid))
print("numeric valid ratio:", len(mobility_numeric_valid) / len(mobility_numeric) if len(mobility_numeric) else 0)

# =========================
# 6. 복원된 mobility 합치기
# =========================
mobility_restored = pd.concat(
    [mobility_korean, mobility_numeric_valid],
    ignore_index=True
)

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility_restored[c] = pd.to_numeric(mobility_restored[c], errors="coerce").fillna(0)

mobility_restored["CELL_ID"] = mobility_restored["CELL_ID"].astype(str).str.strip()

print("\n[MOBILITY RESTORED]")
print("rows:", len(mobility_restored))
print("unique CELL_ID:", mobility_restored["CELL_ID"].nunique())
print(mobility_restored[["CELL_ID_BASE_RAW", "CELL_ID", "CELL_X", "CELL_Y", "out_cnt", "in_cnt", "net_cnt"]].head(20))

# =========================
# 7. CELL 단위 mobility 집계
# =========================
cell_mobility = (
    mobility_restored
    .groupby("CELL_ID", as_index=False)
    .agg(
        CELL_X=("CELL_X", "first"),
        CELL_Y=("CELL_Y", "first"),
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())

# =========================
# 8. merge check
# =========================
mobility_ids = set(cell_mobility["CELL_ID"])
mapping_ids = set(mapping["CELL_ID"])
overlap_ids = mobility_ids & mapping_ids

check = pd.DataFrame([{
    "raw_rows": len(mobility),
    "raw_unique_CELL_ID_BASE": mobility["CELL_ID_BASE_RAW"].nunique(),
    "korean_rows": len(mobility_korean),
    "numeric_valid_rows": len(mobility_numeric_valid),
    "numeric_invalid_rows": len(mobility_numeric_invalid),
    "restored_rows": len(mobility_restored),
    "restored_unique_CELL_ID": len(mobility_ids),
    "mapping_unique_CELL_ID": len(mapping_ids),
    "overlap_CELL_ID": len(overlap_ids),
    "overlap_ratio_vs_mobility": len(overlap_ids) / len(mobility_ids) if len(mobility_ids) else 0,
    "overlap_ratio_vs_mapping": len(overlap_ids) / len(mapping_ids) if len(mapping_ids) else 0,
}])

print("\n[MERGE CHECK]")
print(check)

check.to_csv(check_output_path, index=False, encoding="utf-8-sig")

# =========================
# 9. CELL-level merge
# =========================
cell_level = mapping.merge(
    cell_mobility,
    on="CELL_ID",
    how="left",
    suffixes=("", "_mobility")
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_level[c] = pd.to_numeric(cell_level[c], errors="coerce").fillna(0)

# =========================
# 10. 지표 생성
# =========================
cell_level["actual_total_flow_norm"] = normalize_minmax(cell_level["actual_total_flow"])
cell_level["distance_norm"] = normalize_minmax(cell_level["nearest_station_distance_m"])
cell_level["accessibility_norm"] = normalize_minmax(cell_level["station_accessibility_score"])

cell_level["cell_hidden_demand_score"] = (
    cell_level["actual_total_flow_norm"] *
    (1 - cell_level["accessibility_norm"])
)

cell_level["distance_weighted_demand"] = (
    cell_level["actual_total_flow_norm"] *
    cell_level["distance_norm"]
)

# =========================
# 11. 누락 cell 저장
# =========================
mobility_not_in_mapping_ids = mobility_ids - mapping_ids

missing_mobility = cell_mobility[
    cell_mobility["CELL_ID"].isin(mobility_not_in_mapping_ids)
].copy()

missing_mobility = missing_mobility.sort_values(
    "actual_total_flow",
    ascending=False
)

missing_mobility.to_csv(
    missing_output_path,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 12. 저장
# =========================
cell_level.to_csv(output_path, index=False, encoding="utf-8-sig")

top_cells = (
    cell_level
    .sort_values("cell_hidden_demand_score", ascending=False)
    .head(200)
)

top_cells.to_csv(top_output_path, index=False, encoding="utf-8-sig")

# =========================
# 13. 결과 확인
# =========================
print("\n================================================")
print("DONE")
print("================================================")
print("merged:", output_path)
print("top cells:", top_output_path)
print("missing mobility:", missing_output_path)
print("check:", check_output_path)

print("\n[CELL LEVEL SHAPE]")
print(cell_level.shape)

print("\n[KEY METRICS]")
print(cell_level[[
    "actual_total_flow",
    "nearest_station_distance_m",
    "actual_total_flow_norm",
    "distance_norm",
    "accessibility_norm",
    "cell_hidden_demand_score",
    "distance_weighted_demand"
]].describe())

print("\n[TOP CELLS]")
print(top_cells[[
    "CELL_ID",
    "MASTER_STATION_ID",
    "nearest_station",
    "nearest_line",
    "distance_group",
    "actual_total_flow",
    "nearest_station_distance_m",
    "cell_hidden_demand_score",
    "distance_weighted_demand"
]].head(30))

print("\n[TOP MOBILITY NOT IN MAPPING]")
print(missing_mobility.head(30))

read success: CELL_250x250(UTMK_epsg 5179).csv / utf-8-sig
read success: cell_to_nearest_station_VALID_5km_MASTER_ID.csv / utf-8-sig

[CELL MASTER] (207205, 4)
[MAPPING] (85543, 21)
[MOBILITY RAW] (630941, 6)

[ID TYPE CHECK]
rows: 630941
korean CELL_ID rows: 628874
numeric-like rows: 2067
CELL_ID_BASE unique: 28486

[NUMERIC RESTORE CHECK]
numeric rows: 2067
numeric valid rows: 2067
numeric invalid rows: 0
numeric valid ratio: 1.0

[MOBILITY RESTORED]
rows: 630941
unique CELL_ID: 28475
   CELL_ID_BASE_RAW     CELL_ID  CELL_X   CELL_Y  out_cnt  in_cnt  net_cnt
0        가사51259925  가사51259925  751375  1999375      3.5     0.0     -3.5
1        가사56257725  가사56257725  756375  1977375      7.0     0.0     -7.0
2        나바62009925  나바62009925  862125  1899375      0.0     3.5      3.5
3        나사76250950  나사76250950  876375  1909625      3.5     3.5      0.0
4        나사80001575  나사80001575  880125  1915875      3.5     0.0     -3.5
5        나사86001025  나사86001025  886125  1910375      3.5 

In [ ]:
# ============================================================
# DATA COMPARISON STEP
# 목적:
# 1) 역세권 mapping 안/밖 생활이동 비교
# 2) 거리구간별 생활이동 비교
# 3) 역별 주변 생활이동량 집계
# 4) TOP hidden / not-covered mobility cell 추출
# ============================================================

import os
import pandas as pd
import numpy as np

# =========================
# 0. 경로
# =========================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\comparison_analysis"
os.makedirs(output_dir, exist_ok=True)

cell_level_path = r"C:\Users\82108\Desktop\새 폴더 (8)\cell_level_analysis_mixed\CELL_LEVEL_MIXED_merged.csv"
missing_path = r"C:\Users\82108\Desktop\새 폴더 (8)\cell_level_analysis_mixed\CELL_LEVEL_MIXED_mobility_not_in_mapping.csv"

output_cell_summary = os.path.join(output_dir, "01_cell_mapping_coverage_summary.csv")
output_distance_summary = os.path.join(output_dir, "02_distance_group_mobility_summary.csv")
output_station_summary = os.path.join(output_dir, "03_station_surrounding_mobility_summary.csv")
output_top_hidden = os.path.join(output_dir, "04_top_hidden_demand_cells.csv")
output_top_missing = os.path.join(output_dir, "05_top_mobility_not_in_station_mapping.csv")

# =========================
# 1. 로드
# =========================

cell_level = pd.read_csv(cell_level_path, encoding="utf-8-sig")
missing = pd.read_csv(missing_path, encoding="utf-8-sig")

print("[CELL LEVEL]")
print(cell_level.shape)
print(cell_level.head())

print("\n[MISSING MOBILITY]")
print(missing.shape)
print(missing.head())

# =========================
# 2. 매핑 안/밖 coverage 비교
# =========================

mapped_flow = cell_level["actual_total_flow"].sum()
missing_flow = missing["actual_total_flow"].sum()

coverage_summary = pd.DataFrame([{
    "mapped_cell_count": (cell_level["actual_total_flow"] > 0).sum(),
    "mapped_total_flow": mapped_flow,
    "missing_cell_count": len(missing),
    "missing_total_flow": missing_flow,
    "total_flow": mapped_flow + missing_flow,
    "mapped_flow_ratio": mapped_flow / (mapped_flow + missing_flow) if (mapped_flow + missing_flow) > 0 else np.nan,
    "missing_flow_ratio": missing_flow / (mapped_flow + missing_flow) if (mapped_flow + missing_flow) > 0 else np.nan
}])

coverage_summary.to_csv(output_cell_summary, index=False, encoding="utf-8-sig")

print("\n[COVERAGE SUMMARY]")
print(coverage_summary)

# =========================
# 3. 거리구간별 생활이동 비교
# =========================

distance_summary = (
    cell_level
    .groupby("distance_group", dropna=False)
    .agg(
        cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        total_flow=("actual_total_flow", "sum"),
        mean_flow=("actual_total_flow", "mean"),
        median_flow=("actual_total_flow", "median"),
        max_flow=("actual_total_flow", "max"),
        mean_distance=("nearest_station_distance_m", "mean"),
        mean_hidden_score=("cell_hidden_demand_score", "mean"),
        max_hidden_score=("cell_hidden_demand_score", "max")
    )
    .reset_index()
)

distance_summary["flow_ratio"] = (
    distance_summary["total_flow"] / distance_summary["total_flow"].sum()
)

distance_summary.to_csv(output_distance_summary, index=False, encoding="utf-8-sig")

print("\n[DISTANCE GROUP SUMMARY]")
print(distance_summary)

# =========================
# 4. 역별 주변 생활이동량 요약
# =========================

station_summary = (
    cell_level
    .groupby(["MASTER_STATION_ID", "nearest_station", "nearest_line"], dropna=False)
    .agg(
        mapped_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        station_area_total_flow=("actual_total_flow", "sum"),
        station_area_outflow=("mobility_out_sum", "sum"),
        station_area_inflow=("mobility_in_sum", "sum"),
        station_area_netflow=("mobility_net_sum", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max"),
        mean_hidden_score=("cell_hidden_demand_score", "mean"),
        max_hidden_score=("cell_hidden_demand_score", "max")
    )
    .reset_index()
)

station_summary["active_cell_ratio"] = (
    station_summary["active_cell_count"] / station_summary["mapped_cell_count"]
)

station_summary = station_summary.sort_values(
    "station_area_total_flow",
    ascending=False
)

station_summary.to_csv(output_station_summary, index=False, encoding="utf-8-sig")

print("\n[STATION SUMMARY TOP]")
print(station_summary.head(30))

# =========================
# 5. TOP hidden demand cells
# =========================

top_hidden = (
    cell_level
    .sort_values("cell_hidden_demand_score", ascending=False)
    .head(300)
)

top_hidden.to_csv(output_top_hidden, index=False, encoding="utf-8-sig")

print("\n[TOP HIDDEN CELLS]")
print(top_hidden[[
    "CELL_ID",
    "MASTER_STATION_ID",
    "nearest_station",
    "nearest_line",
    "distance_group",
    "actual_total_flow",
    "nearest_station_distance_m",
    "cell_hidden_demand_score",
    "distance_weighted_demand"
]].head(30))

# =========================
# 6. TOP mobility not in station mapping
# =========================

top_missing = (
    missing
    .sort_values("actual_total_flow", ascending=False)
    .head(300)
)

top_missing.to_csv(output_top_missing, index=False, encoding="utf-8-sig")

print("\n[TOP MOBILITY NOT IN MAPPING]")
print(top_missing[[
    "CELL_ID",
    "CELL_X",
    "CELL_Y",
    "actual_total_flow",
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum"
]].head(30))

# =========================
# 7. 저장 위치
# =========================

print("\n================================================")
print("DONE")
print("================================================")
print("saved folder:", output_dir)
print(output_cell_summary)
print(output_distance_summary)
print(output_station_summary)
print(output_top_hidden)
print(output_top_missing)

read success: cell_to_nearest_station_VALID_5km_MASTER_ID.csv / utf-8-sig

[MAPPING RAW]
(85543, 21)
['MASTER_STATION_ID', 'CELL_ID', 'CELL_X', 'CELL_Y', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group', 'master_station_name', 'master_line']
   MASTER_STATION_ID     CELL_ID  CELL_X   CELL_Y nearest_station  \
0               3121  다사31254225  931375  1942375              동수   
1               1761  다사33752875  933875  1928875              정왕   
2               2614  다사53005725  953125  1957375             독바위   
3                426  다사53255000  953375  1950125             서울역   
4               1918  다아58500350  958625  2003625              전곡   

  nearest_line  station_lat  station_lon  station_x_5179  station_y_5179  ..

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# 0. 경로
# ============================================================

output_dir = r"C:\Users\82108\Desktop\새 폴더 (8)\masterid_subway_compare_20170101"
os.makedirs(output_dir, exist_ok=True)

# 이미 만든 MASTER_ID 기반 cell-station mapping
mapping_path = r"C:\Users\82108\Desktop\새 폴더 (8)\station_mapping_final\cell_to_nearest_station_VALID_5km_MASTER_ID.csv"

# mobility
mobility_path = r"C:\Users\82108\Desktop\새 폴더 (7)\mobility_cell_output\202301__202301__PURPOSE_forn_250M_20230102_cell.parquet"

# 비교할 subway parquet
subway_path = r"C:\Users\82108\Desktop\새 폴더 (8)\od_output\od_output\202002_TBDM_TRNSIT_OD_SUBWAY_20191002_station.parquet"

# 저장
cell_level_out = os.path.join(output_dir, "01_cell_level_with_masterid.csv")
station_mobility_out = os.path.join(output_dir, "02_station_mobility_masterid.csv")
subway_station_out = os.path.join(output_dir, "03_subway_station_activity.csv")
final_out = os.path.join(output_dir, "04_masterid_mobility_vs_subway.csv")
check_out = os.path.join(output_dir, "00_id_overlap_check.csv")


# ============================================================
# 1. 유틸
# ============================================================

def read_csv_safe(path):
    for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"read success: {os.path.basename(path)} / {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError(f"CSV 읽기 실패: {path}")


def normalize_minmax(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    if len(s) == 0 or s.max() == s.min():
        return pd.Series(0, index=s.index)
    return (s - s.min()) / (s.max() - s.min())


def is_korean_cell_id(series):
    return series.astype(str).str.contains(r"[가-힣]", regex=True)


# ============================================================
# 2. MASTER_ID mapping 로드
# ============================================================

mapping = read_csv_safe(mapping_path)

print("\n[MAPPING RAW]")
print(mapping.shape)
print(mapping.columns.tolist())
print(mapping.head())

mapping["CELL_ID"] = mapping["CELL_ID"].astype(str).str.strip()
mapping["MASTER_STATION_ID"] = pd.to_numeric(mapping["MASTER_STATION_ID"], errors="coerce")

mapping = mapping.dropna(subset=["MASTER_STATION_ID"]).copy()
mapping["MASTER_STATION_ID"] = mapping["MASTER_STATION_ID"].astype(int)

mapping = (
    mapping
    .sort_values(["CELL_ID", "nearest_station_distance_m", "MASTER_STATION_ID"])
    .drop_duplicates(subset=["CELL_ID"])
    .copy()
)

print("\n[MAPPING CLEAN]")
print("rows:", len(mapping))
print("unique CELL_ID:", mapping["CELL_ID"].nunique())
print("unique MASTER_STATION_ID:", mapping["MASTER_STATION_ID"].nunique())


# ============================================================
# 3. mobility 로드
#    핵심: 한글 CELL_ID는 그대로 사용
#    숫자형은 여기서는 제외
# ============================================================

mobility = pd.read_parquet(mobility_path)

print("\n[MOBILITY RAW]")
print(mobility.shape)
print(mobility.head())

mobility["CELL_ID_BASE_RAW"] = mobility["CELL_ID_BASE"].astype(str).str.strip()

korean_mask = is_korean_cell_id(mobility["CELL_ID_BASE_RAW"])

print("\n[MOBILITY ID TYPE]")
print("rows:", len(mobility))
print("korean CELL_ID rows:", korean_mask.sum())
print("numeric-like rows:", (~korean_mask).sum())
print("unique raw CELL_ID_BASE:", mobility["CELL_ID_BASE_RAW"].nunique())

mobility = mobility[korean_mask].copy()
mobility["CELL_ID"] = mobility["CELL_ID_BASE_RAW"]

for c in ["out_cnt", "in_cnt", "net_cnt"]:
    mobility[c] = pd.to_numeric(mobility[c], errors="coerce").fillna(0)


# ============================================================
# 4. CELL 단위 mobility 집계
# ============================================================

cell_mobility = (
    mobility
    .groupby("CELL_ID", as_index=False)
    .agg(
        mobility_out_sum=("out_cnt", "sum"),
        mobility_in_sum=("in_cnt", "sum"),
        mobility_net_sum=("net_cnt", "sum"),
        mobility_record_count=("datetime", "count")
    )
)

cell_mobility["actual_total_flow"] = (
    cell_mobility["mobility_out_sum"] + cell_mobility["mobility_in_sum"]
)

cell_mobility["actual_abs_net_flow"] = cell_mobility["mobility_net_sum"].abs()

print("\n[CELL MOBILITY]")
print(cell_mobility.shape)
print(cell_mobility.head())


# ============================================================
# 5. CELL mobility + MASTER_ID mapping merge
# ============================================================

cell_level = mapping.merge(
    cell_mobility,
    on="CELL_ID",
    how="left"
)

for c in [
    "mobility_out_sum",
    "mobility_in_sum",
    "mobility_net_sum",
    "mobility_record_count",
    "actual_total_flow",
    "actual_abs_net_flow"
]:
    cell_level[c] = pd.to_numeric(cell_level[c], errors="coerce").fillna(0)

cell_level.to_csv(cell_level_out, index=False, encoding="utf-8-sig")

print("\n[CELL LEVEL]")
print(cell_level.shape)
print("active cells:", (cell_level["actual_total_flow"] > 0).sum())


# ============================================================
# 6. MASTER_STATION_ID 기준 주변 mobility 집계
# ============================================================

station_mobility = (
    cell_level
    .groupby(["MASTER_STATION_ID", "nearest_station", "nearest_line"], as_index=False)
    .agg(
        mapped_cell_count=("CELL_ID", "count"),
        active_cell_count=("actual_total_flow", lambda x: (x > 0).sum()),
        surrounding_total_flow=("actual_total_flow", "sum"),
        surrounding_outflow=("mobility_out_sum", "sum"),
        surrounding_inflow=("mobility_in_sum", "sum"),
        surrounding_netflow=("mobility_net_sum", "sum"),
        mean_cell_flow=("actual_total_flow", "mean"),
        median_cell_flow=("actual_total_flow", "median"),
        max_cell_flow=("actual_total_flow", "max"),
        mean_distance_m=("nearest_station_distance_m", "mean"),
        min_distance_m=("nearest_station_distance_m", "min"),
        max_distance_m=("nearest_station_distance_m", "max")
    )
)

station_mobility["active_cell_ratio"] = (
    station_mobility["active_cell_count"] / station_mobility["mapped_cell_count"]
)

station_mobility.to_csv(station_mobility_out, index=False, encoding="utf-8-sig")

print("\n[STATION MOBILITY]")
print(station_mobility.shape)
print(station_mobility.head())


# ============================================================
# 7. subway parquet 로드 + 역별 activity 집계
# ============================================================

subway = pd.read_parquet(subway_path)

print("\n[SUBWAY RAW]")
print(subway.shape)
print(subway.columns.tolist())
print(subway.head())

# STATION_ID 컬럼명 대응
# 만약 subwayid라는 컬럼이면 자동 rename
if "subwayid" in subway.columns and "STATION_ID" not in subway.columns:
    subway = subway.rename(columns={"subwayid": "STATION_ID"})

if "STATION_ID" not in subway.columns:
    raise ValueError("subway parquet에 STATION_ID 또는 subwayid 컬럼이 없음")

subway["STATION_ID"] = pd.to_numeric(subway["STATION_ID"], errors="coerce")

for c in ["od_out_cnt", "od_in_cnt", "od_net_cnt"]:
    if c not in subway.columns:
        raise ValueError(f"subway parquet에 {c} 컬럼이 없음")
    subway[c] = pd.to_numeric(subway[c], errors="coerce").fillna(0)

subway_station = (
    subway
    .dropna(subset=["STATION_ID"])
    .assign(MASTER_STATION_ID=lambda x: x["STATION_ID"].astype(int))
    .groupby("MASTER_STATION_ID", as_index=False)
    .agg(
        subway_out_sum=("od_out_cnt", "sum"),
        subway_in_sum=("od_in_cnt", "sum"),
        subway_net_sum=("od_net_cnt", "sum"),
        subway_record_count=("datetime", "count")
    )
)

subway_station["subway_total_activity"] = (
    subway_station["subway_out_sum"] + subway_station["subway_in_sum"]
)

subway_station.to_csv(subway_station_out, index=False, encoding="utf-8-sig")

print("\n[SUBWAY STATION]")
print(subway_station.shape)
print(subway_station.head())


# ============================================================
# 8. ID overlap check
# ============================================================

mobility_ids = set(station_mobility["MASTER_STATION_ID"].astype(int))
subway_ids = set(subway_station["MASTER_STATION_ID"].astype(int))

check = pd.DataFrame([{
    "station_mobility_ids": len(mobility_ids),
    "subway_ids": len(subway_ids),
    "overlap_ids": len(mobility_ids & subway_ids),
    "mobility_only_ids": len(mobility_ids - subway_ids),
    "subway_only_ids": len(subway_ids - mobility_ids),
    "overlap_ratio_vs_mobility": len(mobility_ids & subway_ids) / len(mobility_ids) if mobility_ids else 0,
    "overlap_ratio_vs_subway": len(mobility_ids & subway_ids) / len(subway_ids) if subway_ids else 0,
}])

check.to_csv(check_out, index=False, encoding="utf-8-sig")

print("\n[ID OVERLAP CHECK]")
print(check)
print("overlap sample:", sorted(list(mobility_ids & subway_ids))[:50])
print("mobility only sample:", sorted(list(mobility_ids - subway_ids))[:50])
print("subway only sample:", sorted(list(subway_ids - mobility_ids))[:50])


# ============================================================
# 9. 최종 비교: MASTER_STATION_ID == subway STATION_ID
# ============================================================

final = station_mobility.merge(
    subway_station,
    on="MASTER_STATION_ID",
    how="inner"
)

print("\n[FINAL MERGE]")
print(final.shape)
print(final.head())


# ============================================================
# 10. 비교 지표
# ============================================================

final["mobility_norm"] = normalize_minmax(final["surrounding_total_flow"])
final["subway_norm"] = normalize_minmax(final["subway_total_activity"])

final["mobility_minus_subway_gap"] = (
    final["mobility_norm"] - final["subway_norm"]
)

final["hidden_demand_score"] = (
    final["mobility_norm"] * (1 - final["subway_norm"])
)

final["subway_overrepresentation_score"] = (
    final["subway_norm"] * (1 - final["mobility_norm"])
)

final["distance_norm"] = normalize_minmax(final["mean_distance_m"])

final["distance_weighted_hidden_demand"] = (
    final["mobility_norm"] * final["distance_norm"]
)

# 분류
q_hidden = final["hidden_demand_score"].quantile(0.90) if len(final) else 0
q_over = final["subway_overrepresentation_score"].quantile(0.90) if len(final) else 0

conditions = [
    final["hidden_demand_score"] >= q_hidden,
    final["subway_overrepresentation_score"] >= q_over,
]

choices = [
    "mobility_high_subway_low",
    "subway_high_mobility_low",
]

final["station_type"] = np.select(conditions, choices, default="normal")

final = final.sort_values("hidden_demand_score", ascending=False)

final.to_csv(final_out, index=False, encoding="utf-8-sig")


# ============================================================
# 11. 결과 확인
# ============================================================

print("\n================================================")
print("DONE")
print("================================================")
print("check:", check_out)
print("cell level:", cell_level_out)
print("station mobility:", station_mobility_out)
print("subway station:", subway_station_out)
print("final:", final_out)

print("\n[FINAL SHAPE]")
print(final.shape)

print("\n[KEY METRICS]")
print(final[[
    "surrounding_total_flow",
    "subway_total_activity",
    "mobility_norm",
    "subway_norm",
    "mobility_minus_subway_gap",
    "hidden_demand_score",
    "subway_overrepresentation_score",
    "mean_distance_m"
]].describe())

print("\n[STATION TYPE COUNTS]")
print(final["station_type"].value_counts())

print("\n[TOP HIDDEN DEMAND]")
print(final[[
    "MASTER_STATION_ID",
    "nearest_station",
    "nearest_line",
    "mapped_cell_count",
    "active_cell_count",
    "surrounding_total_flow",
    "subway_total_activity",
    "mobility_norm",
    "subway_norm",
    "mobility_minus_subway_gap",
    "hidden_demand_score",
    "station_type"
]].head(30))

read success: cell_to_nearest_station_VALID_5km_MASTER_ID.csv / utf-8-sig

[MAPPING RAW]
(85543, 21)
['MASTER_STATION_ID', 'CELL_ID', 'CELL_X', 'CELL_Y', 'nearest_station', 'nearest_line', 'station_lat', 'station_lon', 'station_x_5179', 'station_y_5179', 'nearest_station_distance_m', 'station_access_250m', 'station_access_500m', 'station_access_1km', 'station_access_2km', 'station_access_5km', 'valid_station_mapping', 'station_accessibility_score', 'distance_group', 'master_station_name', 'master_line']
   MASTER_STATION_ID     CELL_ID  CELL_X   CELL_Y nearest_station  \
0               3121  다사31254225  931375  1942375              동수   
1               1761  다사33752875  933875  1928875              정왕   
2               2614  다사53005725  953125  1957375             독바위   
3                426  다사53255000  953375  1950125             서울역   
4               1918  다아58500350  958625  2003625              전곡   

  nearest_line  station_lat  station_lon  station_x_5179  station_y_5179  ..